In [0]:
%pip install openpyxl

import pandas as pd

BASE = "/Volumes/mvp-base_programação_operacional/bronze/raw_extraction"

In [0]:
spark.sql("USE CATALOG `mvp-base_programação_operacional`")

df_a = spark.table("bronze.sistema_a").toPandas()
df_b = spark.table("bronze.sistema_b").toPandas()
df_c = spark.table("bronze.sistema_c").toPandas()
df_d = spark.table("bronze.sistema_d").toPandas()

print(df_a.shape, df_b.shape, df_c.shape, df_d.shape)
df_a.head()

In [0]:
DE_PARA_TERRITORIO = {
    "sp sp metropolitano": "SP SP METROPOLITANO",
    "sp - metropolitana": "SP SP METROPOLITANO",
    "sao paulo metro": "SP SP METROPOLITANO",
    "sp metropolitano": "SP SP METROPOLITANO",
    "sp sp campinas": "SP SP CAMPINAS",
    "campinas-sp": "SP SP CAMPINAS",
    "sp campinas": "SP SP CAMPINAS",
    "rj rj capital": "RJ RJ CAPITAL",
    "rio de janeiro capital": "RJ RJ CAPITAL",
    "rj capital": "RJ RJ CAPITAL",
    "mg bh metropolitano": "MG BH METROPOLITANO",
    "belo horizonte metro": "MG BH METROPOLITANO",
    "mg bh metro": "MG BH METROPOLITANO",
    "rs poa metropolitano": "RS POA METROPOLITANO",
    "porto alegre metro": "RS POA METROPOLITANO",
    "rs poa metro": "RS POA METROPOLITANO",
}

def normalizar_territorio(valor):
    if pd.isna(valor):
        return None
    chave = str(valor).strip().lower()
    return DE_PARA_TERRITORIO.get(chave, f"NAO_MAPEADO: {valor}")

df_a["territorio_normalizado"] = df_a["TERRITORIO"].apply(normalizar_territorio)
df_b["territorio_normalizado"] = df_b["CLUSTER"].apply(normalizar_territorio)
df_c["territorio_normalizado"] = df_c["Territorio_Atend"].apply(normalizar_territorio)
df_d["territorio_normalizado"] = df_d["Cluster_Varejo"].apply(normalizar_territorio)


def classificar_pf_pj(documento):
    if pd.isna(documento):
        return None
    tamanho = len(str(int(documento))) if str(documento).replace('.', '').isdigit() else len(str(documento))
    return "PJ" if tamanho > 11 else "PF"

df_a["tipo_documento"] = df_a["CNPJ_CPF"].apply(classificar_pf_pj)
df_b["tipo_documento"] = df_b["DOCUMENTO"].apply(classificar_pf_pj)
df_c["tipo_documento"] = df_c["CNPJ"].apply(classificar_pf_pj)
# Sistema D não tem coluna de documento nos dados sintéticos


DE_PARA_STATUS = {
    "AGENDADO INFRA": "agendado",
    "AGENDADO VISTORIA": "agendado",
    "AGENDADO": "agendado",
    "OS AGENDADA": "agendado",
    "PENDENTE POS VENDAS": "atrasado",
    "EM FOLLOW-UP": "atrasado",
    "ABERTA": "atrasado",
    "PENDENTE": "atrasado",
    "AGENDAR INFRA": "oportunidade",
    "SEM DATA": "oportunidade",
    "AGUARDANDO AGENDA": "oportunidade",
}

df_a["status_unificado"] = df_a["TIPO_PENDENCIA"].map(DE_PARA_STATUS)
df_b["status_unificado"] = df_b["STATUS"].map(DE_PARA_STATUS)
df_c["status_unificado"] = df_c["Situacao_OS"].map(DE_PARA_STATUS)
df_d["status_unificado"] = df_d["Status"].map(DE_PARA_STATUS)

print("PF/PJ - valores únicos:")
print(df_a["tipo_documento"].value_counts())
print("\nStatus não mapeado em cada sistema:")
for nome, df, col in [("A", df_a, "TIPO_PENDENCIA"), ("B", df_b, "STATUS"), ("C", df_c, "Situacao_OS"), ("D", df_d, "Status")]:
    faltando = df[df["status_unificado"].isna()][col].unique()
    print(f"  Sistema {nome}: {faltando}")

In [0]:
df_a_std = df_a.rename(columns={
    "NUM_CONTRATO": "id_contrato",
    "CNPJ_CPF": "documento",
    "TIPO_ATIVIDADE": "tipo_atividade",
    "VALOR_RECORRENTE": "valor_recorrente",
    "DATA_AGENDAMENTO": "data_agendamento",
})
df_a_std["sistema_origem"] = "A"

df_b_std = df_b.rename(columns={
    "ID_CONTRATO": "id_contrato",
    "DOCUMENTO": "documento",
    "SERVICO_ABERTURA": "tipo_atividade",
    "DATA_ABERTURA": "data_agendamento",
})
df_b_std["valor_recorrente"] = None
df_b_std["sistema_origem"] = "B"

df_c_std = df_c.rename(columns={
    "Contrato": "id_contrato",
    "CNPJ": "documento",
    "Tipo_Atendimento": "tipo_atividade",
    "Data_Agenda": "data_agendamento",
})
df_c_std["valor_recorrente"] = None
df_c_std["sistema_origem"] = "C"

df_d_std = df_d.rename(columns={
    "Numero_Contrato": "id_contrato",
    "Data_Agendamento": "data_agendamento",
})
df_d_std["documento"] = None
df_d_std["tipo_documento"] = None
df_d_std["tipo_atividade"] = None
df_d_std["valor_recorrente"] = None
df_d_std["sistema_origem"] = "D"

COLUNAS_FINAIS = [
    "id_contrato", "documento", "tipo_documento", "territorio_normalizado",
    "status_unificado", "tipo_atividade", "valor_recorrente", "data_agendamento",
    "sistema_origem", "_data_ingestao", "_arquivo_origem",
]

df_silver = pd.concat([
    df_a_std.reindex(columns=COLUNAS_FINAIS),
    df_b_std.reindex(columns=COLUNAS_FINAIS),
    df_c_std.reindex(columns=COLUNAS_FINAIS),
    df_d_std.reindex(columns=COLUNAS_FINAIS),
], ignore_index=True)

contagem_por_contrato = df_silver["id_contrato"].value_counts()
contratos_replicados = contagem_por_contrato[contagem_por_contrato > 1].index
df_silver["flag_replicado_multissistema"] = df_silver["id_contrato"].isin(contratos_replicados)

print(f"Total de linhas unificadas: {len(df_silver)}")
print(f"Contratos replicados entre sistemas: {df_silver['flag_replicado_multissistema'].sum()}")
df_silver.head(10)

In [0]:
DE_PARA_TIPO_ATIVIDADE = {
    "AGENDADO INFRA": "instalação",
    "AGENDADO VISTORIA": "vistoria",
    "INSTALACAO FIBRA": "instalação",
    "VISTORIA FIBRA": "vistoria",
    "INSTALACAO (OS)": "instalação",
    "VISTORIA (OS)": "vistoria",
}

def normalizar_tipo_atividade(valor):
    if pd.isna(valor):
        return None
    chave = str(valor).strip().upper()
    return DE_PARA_TIPO_ATIVIDADE.get(chave, f"NAO_MAPEADO: {valor}")

df_silver["tipo_atividade_normalizado"] = df_silver["tipo_atividade"].apply(normalizar_tipo_atividade)

print(df_silver["tipo_atividade_normalizado"].value_counts(dropna=False))

In [0]:
spark.createDataFrame(df_silver).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.agendamentos_unificado")

print("Tabela silver.agendamentos_unificado gravada com sucesso")
print(f"Total de linhas: {len(df_silver)}")